# 🌿 Chatbot Cengkeh — Uji Backend

Notebook ini mendokumentasikan **apa yang terjadi di backend** ketika pengguna berinteraksi dengan Chatbot Cengkeh.

**Prasyarat:**
- Aplikasi Flask berjalan di `http://<ip>:5000`
- Database sudah terisi dengan knowledge base Cengkeh (100 QA, follow-up, dll)

**Yang akan kita bahas:**
1. **Health check** — pastikan API bisa diakses
2. **Tokenisasi** — bagaimana pertanyaan pengguna dipecah menjadi kata kunci
3. **Pencarian QA** — menemukan pertanyaan yang cocok
4. **Detail QA** — jawaban, sumber, dan follow-up
5. **Alur diagnostik** — traversal pohon pertanyaan lanjutan
6. **Trace lengkap** — perjalanan pengguna dari awal sampai akhir

In [9]:
import requests
from IPython.display import display, Markdown, JSON

BASE_URL = "http://web:5000"


def show_json(data):
    """Tampilkan dict sebagai JSON yang rapi."""
    display(JSON(data, expanded=True))

## 1. Health Check

Pertama, kita pastikan Flask API berjalan dan bisa diakses.

**Endpoint:** `GET /api/health`

In [10]:
r = requests.get(f"{BASE_URL}/api/health")
print(f"Status code: {r.status_code}")
show_json(r.json())

Status code: 200


<IPython.core.display.JSON object>

In [ ]:
## 2. Tokenisasi — Memecah Pertanyaan Pengguna

Sebelum mencari jawaban, backend **memecah pertanyaan pengguna** menjadi token (kata kunci) yang bersih.

**Proses yang terjadi:**

1. **Lowercase** — semua huruf diubah menjadi kecil
2. **Split** — dipisahkan berdasarkan spasi
3. **Strip punctuation** — tanda baca di ujung kata dihapus (`?`, `!`, `,`, dll)
4. **Filter pendek** — kata dengan panjang < 3 karakter dibuang
5. **Filter stopword** — kata umum Bahasa Indonesia dibuang (yang, di, ke, cengkeh, dll)
6. **Deduplikasi** — kata yang sama hanya diambil sekali

Hasilnya: **daftar kata kunci** yang bermakna untuk pencarian.

**Endpoint:** `GET /api/search?q=<pertanyaan>`

In [3]:
import re

# Minta pengguna memasukkan pertanyaan
print("=" * 70)
pertanyaan = input("❓ Silakan ketik pertanyaan Anda tentang cengkeh: ").strip()
print("=" * 70)
print()

# Jika kosong, pakai default
if not pertanyaan:
    pertanyaan = "Bagaimana agar cengkeh dapat bertumbuh dengan baik"
    print(f"⚠️  Tidak ada input. Menggunakan default: '{pertanyaan}'")
    print()

# Panggil API
r = requests.get(f"{BASE_URL}/api/search", params={"q": pertanyaan})
data = r.json()

# Tampilkan hasil
print(f"❓ Pertanyaan          : {data['query']}")
print(f"🔤 Token yang tersisa  : {data['tokens']}")
print(f"📊 Jumlah token        : {len(data['tokens'])}")
print(f"🎯 Total match         : {data['total']}")
print()

# Perbandingan sebelum vs sesudah filter
sebelum = pertanyaan.lower().split()
print(f"Sebelum filter ({len(sebelum)} kata): {sebelum}")
print(f"Sesudah filter ({len(data['tokens'])} kata): {data['tokens']}")

❓ Silakan ketik pertanyaan Anda tentang cengkeh:  Kapan cengkeh bisa dipanen?



❓ Pertanyaan          : Kapan cengkeh bisa dipanen?
🔤 Token yang tersisa  : ['dipanen']
📊 Jumlah token        : 1
🎯 Total match         : 1

Sebelum filter (4 kata): ['kapan', 'cengkeh', 'bisa', 'dipanen?']
Sesudah filter (1 kata): ['dipanen']


## 3. Pencarian QA — Menemukan Pertanyaan yang Cocok

Setelah token terbentuk, backend **mencari pertanyaan yang cocok** di database.

**Proses yang terjadi:**

1. **Cari di `tb_keyword`** — token harus *sama persis* dengan kata kunci
   - Setiap match diberi skor **×3** (bobot tinggi, karena keyword sudah dikurasi manual)
2. **Cari di `tb_variasi`** — token harus *terkandung* dalam variasi pertanyaan
   - Setiap match diberi skor **×1** (bobot rendah, karena bisa false positive)
3. **Jumlahkan skor** per QA
4. **Ranking** berdasarkan skor tertinggi
5. **Ambil top-N** (default 3)

Hasilnya: **daftar QA yang paling relevan**, siap ditampilkan ke pengguna.

**Endpoint:** `GET /api/search?q=<pertanyaan>`

In [4]:
print(f"🔍 Pencarian untuk: {data['query']}")
print(f"🔤 Menggunakan token: {data['tokens']}")
print(f"📊 Total QA yang cocok: {data['total']}")
print(f"🎯 Ditampilkan (top-N): {data['shown']}")
print()
print("─" * 70)
print("Hasil pencarian (diurutkan berdasarkan skor):")
print("─" * 70)

# Tampilkan hasil bernomor
for i, m in enumerate(data['matches'], 1):
    print(f"{i}. [ID {m['id']:>3}] (skor {m['score']}) — {m['type']}")
    print(f"   {m['question']}")
    print()

# Minta user memilih
print("─" * 70)
pilihan = input(f"👉 Pilih nomor (1-{len(data['matches'])}) atau tekan Enter untuk pilihan 1: ").strip()

# Proses pilihan
if not pilihan:
    idx = 0
elif pilihan.isdigit() and 1 <= int(pilihan) <= len(data['matches']):
    idx = int(pilihan) - 1
else:
    print(f"⚠️  Pilihan tidak valid: '{pilihan}'. Menggunakan nomor 1.")
    idx = 0

selected_qa_id = data['matches'][idx]['id']
selected_qa_question = data['matches'][idx]['question']

print()
print(f"✅ Anda memilih:")
print(f"   [ID {selected_qa_id}] {selected_qa_question}")

🔍 Pencarian untuk: Kapan cengkeh bisa dipanen?
🔤 Menggunakan token: ['dipanen']
📊 Total QA yang cocok: 1
🎯 Ditampilkan (top-N): 1

──────────────────────────────────────────────────────────────────────
Hasil pencarian (diurutkan berdasarkan skor):
──────────────────────────────────────────────────────────────────────
1. [ID  71] (skor 1) — diagnostic
   Kapan cengkeh biasanya dipanen?

──────────────────────────────────────────────────────────────────────


👉 Pilih nomor (1-1) atau tekan Enter untuk pilihan 1:  1



✅ Anda memilih:
   [ID 71] Kapan cengkeh biasanya dipanen?


## 4. Detail QA — Jawaban, Sumber, dan Follow-up

Setelah pengguna memilih QA dari daftar pencarian, backend mengambil **detail lengkap**:

- **Pertanyaan** dan **jawaban**
- **Sumber** — dokumen Kementan yang relevan (via `tb_pertanyaan_sumber`)
- **QA terkait** — pertanyaan lain yang berhubungan (via `tb_relasi`)
- **Follow-up pertama** — jika QA bertipe `diagnostic`, sertakan pertanyaan lanjutan beserta opsinya

**Endpoint:** `GET /api/qa/<id>`

In [5]:
# Gunakan 'selected_qa_id' dari cell sebelumnya
qa_id = selected_qa_id

print(f"📥 Mengambil detail untuk QA ID {qa_id}...")
print()

r = requests.get(f"{BASE_URL}/api/qa/{qa_id}")
qa = r.json()

# Header
print("─" * 70)
print(f"❓ PERTANYAAN")
print("─" * 70)
print(qa['question'])
print()

# Jawaban
print("─" * 70)
print(f"💡 JAWABAN")
print("─" * 70)
print(qa['answer'])
print()

# Tipe
print(f"📌 Tipe QA: {qa['type']}")
print()

# Sumber
if qa['sources']:
    print("─" * 70)
    print(f"📚 SUMBER ({len(qa['sources'])})")
    print("─" * 70)
    for s in qa['sources']:
        print(f"   • [{s['kode']}] {s['judul']}")
    print()

# QA Terkait
if qa['related']:
    print("─" * 70)
    print(f"🔗 QA TERKAIT ({len(qa['related'])})")
    print("─" * 70)
    for r in qa['related']:
        print(f"   • [ID {r['id']}] {r['question']}")
    print()

# Follow-up
if qa['follow_up']:
    fu = qa['follow_up']
    print("─" * 70)
    print(f"🌳 FOLLOW-UP (urutan {fu['urutan']})")
    print("─" * 70)
    print(f"   {fu['question']}")
    print()
    print("   Opsi jawaban:")
    for o in fu['options']:
        print(f"     [{o['id']}] {o['label']}")
    print()
else:
    print("─" * 70)
    print("🏁 TIDAK ADA FOLLOW-UP — QA ini bersifat terminal.")
    print("─" * 70)

📥 Mengambil detail untuk QA ID 71...

──────────────────────────────────────────────────────────────────────
❓ PERTANYAAN
──────────────────────────────────────────────────────────────────────
Kapan cengkeh biasanya dipanen?

──────────────────────────────────────────────────────────────────────
💡 JAWABAN
──────────────────────────────────────────────────────────────────────
Cengkeh dipanen saat kuncup bunga mencapai kematangan optimal — biasanya 1–2 minggu setelah kuncup berubah warna dari hijau menjadi kemerahan. Di Indonesia, musim panen utama biasanya jatuh pada bulan Juni–Agustus untuk sebagian daerah, tergantung wilayah dan pola iklim setempat. Panen bisa berlangsung 2–4 bulan dengan beberapa kali pemetikan.

📌 Tipe QA: diagnostic

──────────────────────────────────────────────────────────────────────
📚 SUMBER (1)
──────────────────────────────────────────────────────────────────────
   • [S5] Kementan — pedoman teknis budidaya, lingkungan, pembungaan, panen dan pascapanen

─────

## 5. Alur Diagnostik — Traversal Pohon Pertanyaan

Jika QA yang dipilih bertipe **`diagnostic`**, pengguna harus menjawab **follow-up** untuk mempersempit diagnosis.

**Logika backend (`GET /api/next/<option_id>`):**

1. Cari opsi yang dipilih → dapatkan `follow_up_id` dan `urutan`
2. Cek apakah ada **follow-up berikutnya** (`urutan + 1`) untuk QA yang sama
   - **Jika ada** → kembalikan follow-up berikutnya (`type: "follow_up"`)
   - **Jika tidak ada** → kembalikan QA hasil (`type: "matches"`)
3. Pengguna menjawab, proses berulang sampai mencapai QA terminal

**Endpoint:** `GET /api/next/<option_id>`

In [6]:
# Ambil follow-up pertama dari QA yang dipilih di Cell 10
if qa['follow_up'] is None:
    print("⚠️  QA ini tidak memiliki follow-up. Tidak ada alur diagnostik.")
    print("   Jalankan ulang Cell 6, 8, dan 10 dengan QA diagnostic.")
else:
    fu = qa['follow_up']

    print("─" * 70)
    print(f"❓ FOLLOW-UP — {fu['question']}")
    print("─" * 70)
    for i, o in enumerate(fu['options'], 1):
        print(f"  {i}. [{o['id']}] {o['label']}")
    print()

    pilihan = input(f"👉 Pilih nomor (1-{len(fu['options'])}): ").strip()

    if not pilihan or not pilihan.isdigit() or not (1 <= int(pilihan) <= len(fu['options'])):
        print(f"⚠️  Pilihan tidak valid. Menggunakan opsi 1.")
        idx = 0
    else:
        idx = int(pilihan) - 1

    chosen_option = fu['options'][idx]
    chosen_option_id = chosen_option['id']

    print()
    print(f"✅ Anda memilih: [{chosen_option_id}] {chosen_option['label']}")
    print()
    print(f"➡️  Memanggil /api/next/{chosen_option_id}...")
    print()

    # Panggil API
    r = requests.get(f"{BASE_URL}/api/next/{chosen_option_id}")
    next_step = r.json()

    print(f"📥 Response type: {next_step['type']}")
    print()

    if next_step['type'] == 'follow_up':
        nf = next_step['follow_up']
        print("─" * 70)
        print(f"❓ FOLLOW-UP BERIKUTNYA (urutan {nf['urutan']})")
        print("─" * 70)
        print(f"   {nf['question']}")
        print()
        print("   Opsi jawaban:")
        for o in nf['options']:
            print(f"     [{o['id']}] {o['label']}")
    elif next_step['type'] == 'matches':
        print("─" * 70)
        print(f"🎯 HASIL AKHIR — QA yang direkomendasikan ({len(next_step['matches'])})")
        print("─" * 70)
        for m in next_step['matches']:
            print(f"   • [ID {m['id']}] ({m['type']}) {m['question']}")

──────────────────────────────────────────────────────────────────────
❓ FOLLOW-UP — Gejala mana yang paling sesuai?
──────────────────────────────────────────────────────────────────────
  1. [233] Daun/warna berubah
  2. [234] Batang/ranting bermasalah
  3. [235] Tanah/air bermasalah
  4. [236] Tidak tahu



👉 Pilih nomor (1-4):  2



✅ Anda memilih: [234] Batang/ranting bermasalah

➡️  Memanggil /api/next/234...

📥 Response type: follow_up

──────────────────────────────────────────────────────────────────────
❓ FOLLOW-UP BERIKUTNYA (urutan 2)
──────────────────────────────────────────────────────────────────────
   Bagaimana kondisi lokasi?

   Opsi jawaban:
     [237] Kering
     [238] Basah/tergenang
     [239] Normal
     [240] Tidak tahu


### Follow-up 2 — Menjawab dan Melihat Hasil

Setelah follow-up 1 dijawab, pengguna menjawab **follow-up 2** (kondisi lokasi).

Kali ini, karena tidak ada lagi follow-up setelahnya, backend akan mengembalikan **daftar QA yang direkomendasikan** (`type: "matches"`).

In [7]:
# Cek apakah response sebelumnya adalah follow-up
if next_step.get('type') != 'follow_up':
    print("⚠️  Sebelumnya bukan follow-up. Jalankan Cell 12 dengan QA diagnostic terlebih dahulu.")
else:
    fu2 = next_step['follow_up']

    print("─" * 70)
    print(f"❓ FOLLOW-UP 2 — {fu2['question']}")
    print("─" * 70)
    for i, o in enumerate(fu2['options'], 1):
        print(f"  {i}. [{o['id']}] {o['label']}")
    print()

    pilihan2 = input(f"👉 Pilih nomor (1-{len(fu2['options'])}): ").strip()

    if not pilihan2 or not pilihan2.isdigit() or not (1 <= int(pilihan2) <= len(fu2['options'])):
        print(f"⚠️  Pilihan tidak valid. Menggunakan opsi 1.")
        idx2 = 0
    else:
        idx2 = int(pilihan2) - 1

    chosen_option2 = fu2['options'][idx2]
    chosen_option2_id = chosen_option2['id']

    print()
    print(f"✅ Anda memilih: [{chosen_option2_id}] {chosen_option2['label']}")
    print()
    print(f"➡️  Memanggil /api/next/{chosen_option2_id}...")
    print()

    # Panggil API
    r = requests.get(f"{BASE_URL}/api/next/{chosen_option2_id}")
    final_step = r.json()

    print(f"📥 Response type: {final_step['type']}")
    print()

    if final_step['type'] == 'matches':
        print("─" * 70)
        print(f"🎯 HASIL AKHIR — {len(final_step['matches'])} QA direkomendasikan")
        print("─" * 70)
        for m in final_step['matches']:
            print(f"   • [ID {m['id']}] ({m['type']}) {m['question']}")
        print()
        print("Alur diagnostik selesai. Pengguna dapat memilih salah satu QA di atas.")
    else:
        print("⚠️  Masih ada follow-up. Ini tidak terduga untuk data Cengkeh.")
        nf = final_step['follow_up']
        print(f"   {nf['question']}")

──────────────────────────────────────────────────────────────────────
❓ FOLLOW-UP 2 — Bagaimana kondisi lokasi?
──────────────────────────────────────────────────────────────────────
  1. [237] Kering
  2. [238] Basah/tergenang
  3. [239] Normal
  4. [240] Tidak tahu



👉 Pilih nomor (1-4):  2



✅ Anda memilih: [238] Basah/tergenang

➡️  Memanggil /api/next/238...

📥 Response type: matches

──────────────────────────────────────────────────────────────────────
🎯 HASIL AKHIR — 2 QA direkomendasikan
──────────────────────────────────────────────────────────────────────
   • [ID 73] (solution) Jika kuncup cengkeh masih hijau, apa yang dilakukan?
   • [ID 99] (solution) Wadah seperti apa untuk menyimpan cengkeh?

Alur diagnostik selesai. Pengguna dapat memilih salah satu QA di atas.


## 6. Loop Diagnostik — Kenapa Alur Kembali ke Pemilihan QA

Setelah follow-up terakhir dijawab, backend mengembalikan **daftar QA hasil** (`type: matches`). Di sinilah **loop** dimulai.

### Diagram Alur

```
START
  |
  v
[User ketik pertanyaan] --> [/api/search]
  |
  v
[Daftar QA rekomendasi]
  |
  v
[User pilih QA]
  |
  v
[/api/qa/<id>] --> Cek tipe QA
  |
  +---> DIAGNOSTIC (punya follow-up)
  |       |
  |       v
  |     [Tampilkan follow-up]
  |       |
  |       v
  |     [User jawab] --> [/api/next/<option_id>]
  |       |
  |       v
  |     [Lebih banyak follow-up?]
  |       |
  |       +---> YA  --> (ulangi follow-up berikutnya)
  |       |
  |       +---> TIDAK --> [QA hasil]
  |                        |
  |                        v
  |                    (kembali ke "User pilih QA")
  |
  +---> INFORMATIONAL atau SOLUTION
          |
          v
        [Tampilkan jawaban]
          |
          v
        SELESAI (terminal)
```

### Kesimpulan

- **`diagnostic`** QA → selalu punya follow-up → arahkan user menjawab → hasilkan lebih banyak QA → **loop**
- **`informational`** atau **`solution`** QA → **terminal** → alur berhenti di sini, tampilkan jawaban
- **Loop** terus berjalan sampai user memilih QA yang **terminal**

Di cell berikutnya, kita buktikan **loop** ini dengan contoh konkret.

In [8]:
# ==========================================================================
# TRACE LENGKAP — dengan LOOP yang terlihat
# ==========================================================================

def cetak_header(nomor, judul, karakter="█"):
    """Cetak header yang rapi."""
    print()
    print(karakter * 70)
    print(f"{karakter}  {nomor} — {judul}")
    print(karakter * 70)


def label_tipe(tipe):
    """Beri label emoji untuk tipe QA."""
    return {
        'diagnostic': '🌳 DIAGNOSTIC (bisa bercabang)',
        'informational': '📖 INFORMATIONAL (terminal)',
        'solution': '✅ SOLUTION (terminal)',
    }.get(tipe, tipe)


# Mulai dengan pertanyaan pengguna
pertanyaan_demo = "Kenapa daun cengkeh saya menguning?"

cetak_header("MULAI", f"Mengajukan pertanyaan: \"{pertanyaan_demo}\"", "═")

r = requests.get(f"{BASE_URL}/api/search", params={"q": pertanyaan_demo})
search_data = r.json()

print(f"🔤 Token       : {search_data['tokens']}")
print(f"📊 Total match : {search_data['total']}")
print()
print("QA yang direkomendasikan:")
for i, m in enumerate(search_data['matches'], 1):
    print(f"  {i}. [ID {m['id']}] ({m['score']}) {m['question']}")


# === LOOP: mulai dari QA pilihan user ===
loop_count = 1
MAX_LOOP = 5   # batas aman agar tidak infinite

# Pilihan pertama: ambil top match
current_qa_id = search_data['matches'][0]['id']

while loop_count <= MAX_LOOP:
    cetak_header(f"LOOP #{loop_count}", f"Memilih QA ID {current_qa_id}")

    # Ambil detail QA
    r = requests.get(f"{BASE_URL}/api/qa/{current_qa_id}")
    qa = r.json()

    print(f"❓ {qa['question']}")
    print(f"📌 Tipe: {label_tipe(qa['type'])}")
    print()

    # ---- CEK TERMINAL ----
    if qa['type'] in ('informational', 'solution') or qa['follow_up'] is None:
        print("─" * 70)
        print("💡 JAWABAN:")
        print(f"   {qa['answer'][:250]}...")
        print()
        print(f"📚 Sumber: {[s['kode'] for s in qa['sources']]}")
        print()
        cetak_header("SELESAI", f"QA {qa['type'].upper()} tercapai — loop berhenti", "═")
        print()
        print("✅ Pengguna mendapat jawaban terminal.")
        print("   (Jika ingin bertanya lagi, pengguna bisa mulai dari awal.)")
        break

    # ---- DIAGNOSTIC: proses semua follow-up ----
    print("─" * 70)
    print("🌳 QA ini DIAGNOSTIC — perlu menjawab follow-up.")
    print()

    # Ambil follow-up (bisa 1, 2, atau lebih)
    follow_up = qa['follow_up']
    option_id = None
    step = None

    while follow_up is not None:
        # Tampilkan pertanyaan follow-up
        print(f"   ❓ Follow-up (urutan {follow_up['urutan']}): {follow_up['question']}")

        # Pilih opsi pertama (simulasi user)
        opt = follow_up['options'][0]
        print(f"      → Memilih: {opt['label']}")

        # Panggil /api/next
        r = requests.get(f"{BASE_URL}/api/next/{opt['id']}")
        step = r.json()

        if step['type'] == 'follow_up':
            # Masih ada follow-up lanjutan
            follow_up = step['follow_up']
        else:
            # Selesai — dapat daftar QA hasil
            follow_up = None

    # ---- Dapat QA hasil → pilih yang pertama untuk iterasi berikutnya ----
    if step and step['type'] == 'matches' and step['matches']:
        print()
        print(f"   🎯 Hasil: {len(step['matches'])} QA direkomendasikan:")
        for m in step['matches']:
            print(f"      • [ID {m['id']}] ({m['type']}) {m['question']}")

        # LOOP: kembali ke atas dengan QA baru
        current_qa_id = step['matches'][0]['id']
        loop_count += 1
        print()
        print(f"   ↩️  LOOP kembali ke atas dengan QA ID {current_qa_id}...")
    else:
        print("   ⚠️  Tidak ada QA hasil. Loop berhenti.")
        break


if loop_count > MAX_LOOP:
    print()
    print(f"⚠️  Loop mencapai batas maksimum ({MAX_LOOP}). Dihentikan.")


══════════════════════════════════════════════════════════════════════
═  MULAI — Mengajukan pertanyaan: "Kenapa daun cengkeh saya menguning?"
══════════════════════════════════════════════════════════════════════
🔤 Token       : ['daun', 'menguning']
📊 Total match : 10

QA yang direkomendasikan:
  1. [ID 22] (5) Kenapa daun cengkeh menguning?
  2. [ID 23] (5) Daun tua cengkeh menguning, apa yang diperiksa?
  3. [ID 24] (5) Daun muda cengkeh menguning, apa yang diperiksa?

██████████████████████████████████████████████████████████████████████
█  LOOP #1 — Memilih QA ID 22
██████████████████████████████████████████████████████████████████████
❓ Kenapa daun cengkeh menguning?
📌 Tipe: 🌳 DIAGNOSTIC (bisa bercabang)

──────────────────────────────────────────────────────────────────────
🌳 QA ini DIAGNOSTIC — perlu menjawab follow-up.

   ❓ Follow-up (urutan 1): Gejala mana yang paling sesuai?
      → Memilih: Daun/warna berubah
   ❓ Follow-up (urutan 2): Bagaimana kondisi lokasi?
      → M